# Day 62：Capstone Dev Day 4 — 全栈前端 UI 与 FastAPI SSE 流式集成实战

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第六阶段——**Day 62（`v1.0.0-day62`）**！今天我们将使用 `fastapi.testclient.TestClient` 在进程内完成全栈联调验证，无需启动外部端口即可实测：
> 1. **前端页面托管与健康检查**：`GET /` 与 `GET /api/v1/health`；
> 2. **SSE 流式对话与混合 RAG 引用**：`POST /api/v1/capstone/chat/stream`（`status`、`citations`、`token`、`done`）；
> 3. **HITL 高风险审批挂起与恢复**：`POST /api/v1/capstone/approve`；
> 4. **SQLite 持久化与 PII 脱敏审计回查**：`GET /api/v1/capstone/history/{session_id}`。

## 1. 环境初始化与构建隔离的 `TestClient`

In [1]:
import json
import sys
import tempfile
from pathlib import Path
from fastapi.testclient import TestClient

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day62_dir = repo_root / "project_iterations" / "day_62"
if str(day62_dir) not in sys.path:
    sys.path.insert(0, str(day62_dir))

from zhishu_agent.capstone_web import create_app

tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "nb_capstone.db"
app = create_app(db_path=db_path, frontend_dir=day62_dir / "frontend")
client = TestClient(app)

print("已加载模块路径:", day62_dir)
print("健康检查:", client.get("/api/v1/health").json())

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_62
健康检查: {'status': 'ok', 'version': 'v1.0.0-day62', 'service': 'zhishu-capstone'}


/home/antigravity/ab6539763/-AI-Agent/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. 验证 SSE 流式输出与混合 RAG 引用（含 PII 自动脱敏）

In [2]:
def parse_sse(raw_text: str):
    events = []
    for block in raw_text.strip().split("\n\n"):
        if not block.strip():
            continue
        ev_type, data_str = "message", ""
        for line in block.splitlines():
            if line.startswith("event:"):
                ev_type = line[6:].strip()
            elif line.startswith("data:"):
                data_str += line[5:].strip()
        if data_str:
            events.append((ev_type, json.loads(data_str)))
    return events

with client.stream(
    "POST",
    "/api/v1/capstone/chat/stream",
    json={"query": "请分析 2026 Q1 财报营收与毛利率，联系人手机 13812345678", "session_id": "nb-sess-01"},
) as resp:
    assert resp.status_code == 200
    events = parse_sse("".join(resp.iter_text()))

print("收到 SSE 事件序列:", [e[0] for e in events])
citations = next(d["citations"] for name, d in events if name == "citations")
print("命中引用数:", len(citations), "首条引用:", citations[0]["doc_id"], citations[0]["title"])
assert events[-1][1]["status"] == "completed"

收到 SSE 事件序列: ['status', 'status', 'citations', 'status', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'token', 'done']
命中引用数: 3 首条引用: FIN-2026-Q1 智枢科技 2026 年第一季度财报核心摘要


## 3. 验证 HITL 高风险审批挂起、批准恢复与 SQLite 历史回查

In [3]:
with client.stream(
    "POST",
    "/api/v1/capstone/chat/stream",
    json={"query": "申请划拨 20 万元研发专项资金并对外披露公告", "session_id": "nb-sess-hitl"},
) as resp:
    hitl_events = parse_sse("".join(resp.iter_text()))

hitl_payload = next(d for name, d in hitl_events if name == "hitl_required")
print("触发 HITL 挂起单据:", hitl_payload["approval_id"], hitl_payload["action_type"])

approve_res = client.post(
    "/api/v1/capstone/approve",
    json={"approval_id": hitl_payload["approval_id"], "approved": True, "reviewer": "CFO-李明", "comment": "准予划拨"},
).json()
print("审批恢复结果:", approve_res["status"], "|", approve_res["final_message"])

history = client.get("/api/v1/capstone/history/nb-sess-hitl").json()
print("SQLite 持久化消息条数:", len(history["messages"]), "审批记录数:", len(history["approvals"]))
assert len(history["messages"]) == 3
assert history["approvals"][0]["status"] == "approved"
tmp_dir.cleanup()

触发 HITL 挂起单据: apr-60531c3c6c fund_transfer
审批恢复结果: approved | ✅ 【审批已通过】审批人「CFO-李明」已批准操作 `fund_transfer`（单号 apr-60531c3c6c）。备注：准予划拨。任务已恢复并完成闭环。
SQLite 持久化消息条数: 3 审批记录数: 1
